In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ── Load Data ─────────────────────────────────────────
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')

# ── Fix Target ────────────────────────────────────────
if train['Churn'].dtype == object:
    train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0})
train['Churn'] = train['Churn'].astype(int)

print(f'Train : {train.shape} | Test : {test.shape}')
print(f'Churn balance:\n{train["Churn"].value_counts(normalize=True).round(3)}')
print(f'\nDtypes:\n{train.dtypes}')

TARGET   = 'Churn'
ID_COL   = 'id'
N_SPLITS = 10
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# ── Feature Engineering ───────────────────────────────
def feature_engineering(train_df, test_df):
    tr = train_df.copy()
    te = test_df.copy()

    # Force convert all to numeric
    for col in tr.columns:
        if col in [ID_COL, TARGET]: continue
        tr[col] = pd.to_numeric(tr[col], errors='coerce')
    for col in te.columns:
        if col == ID_COL: continue
        te[col] = pd.to_numeric(te[col], errors='coerce')

    # Fill NaN with train median
    for col in tr.columns:
        if col in [ID_COL, TARGET]: continue
        median_val = tr[col].median()
        tr[col]    = tr[col].fillna(median_val)
        te[col]    = te[col].fillna(median_val)

    # Identify cat vs num cols
    cat_cols = [c for c in tr.columns
                if c not in [ID_COL, TARGET]
                and tr[c].nunique() <= 10]
    num_cols = [c for c in tr.columns
                if c not in [ID_COL, TARGET] + cat_cols]

    print(f'Cat cols ({len(cat_cols)}) : {cat_cols}')
    print(f'Num cols ({len(num_cols)}) : {num_cols}')

    # ── OOF Target Encoding (no leakage) ─────────────
    overall_mean = tr[TARGET].mean()
    for col in cat_cols:
        te_col = col + '_te'
        oof_te = np.zeros(len(tr))
        for _, (tr_idx, val_idx) in enumerate(skf.split(tr, tr[TARGET])):
            means         = tr.iloc[tr_idx].groupby(col)[TARGET].mean()
            oof_te[val_idx] = tr.iloc[val_idx][col].map(means).fillna(overall_mean)
        tr[te_col] = oof_te
        full_means = tr.groupby(col)[TARGET].mean()
        te[te_col] = te[col].map(full_means).fillna(overall_mean)

    # ── Label Encode categoricals ─────────────────────
    for col in cat_cols:
        le       = LabelEncoder()
        combined = pd.concat([tr[col], te[col]], axis=0).astype(str)
        le.fit(combined)
        tr[col]  = le.transform(tr[col].astype(str))
        te[col]  = le.transform(te[col].astype(str))

    # ── Numeric Interactions ──────────────────────────
    if 'tenure' in tr.columns and 'MonthlyCharges' in tr.columns:
        for df in [tr, te]:
            df['charge_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)
            df['total_charge_est']  = df['MonthlyCharges'] * df['tenure']
            df['tenure_x_monthly']  = df['tenure'] * df['MonthlyCharges']

    if 'TotalCharges' in tr.columns and 'MonthlyCharges' in tr.columns:
        for df in [tr, te]:
            df['charge_ratio'] = df['TotalCharges'] / (df['MonthlyCharges'] + 1)
            df['charges_diff'] = df['TotalCharges'] - df['total_charge_est']

    if 'tenure' in tr.columns:
        for df in [tr, te]:
            df['tenure_squared']   = df['tenure'] ** 2
            df['tenure_log']       = np.log1p(df['tenure'])
            df['is_new_customer']  = (df['tenure'] <= 3).astype(int)
            df['is_long_customer'] = (df['tenure'] >= 48).astype(int)

    if 'MonthlyCharges' in tr.columns:
        for df in [tr, te]:
            df['monthly_log']   = np.log1p(df['MonthlyCharges'])
            df['is_high_payer'] = (df['MonthlyCharges'] > 65).astype(int)

    if 'TotalCharges' in tr.columns:
        for df in [tr, te]:
            df['total_log'] = np.log1p(df['TotalCharges'])

    # ── Service count ─────────────────────────────────
    service_cols = [c for c in tr.columns if c in [
        'PhoneService','MultipleLines','InternetService',
        'OnlineSecurity','OnlineBackup','DeviceProtection',
        'TechSupport','StreamingTV','StreamingMovies'
    ]]
    if len(service_cols) > 0:
        for df in [tr, te]:
            df['num_services'] = df[service_cols].sum(axis=1)

    # ── Aggregate stats ───────────────────────────────
    all_num = [c for c in tr.columns if c not in [ID_COL, TARGET]]
    for df in [tr, te]:
        df['num_mean'] = df[all_num].mean(axis=1)
        df['num_std']  = df[all_num].std(axis=1)
        df['num_max']  = df[all_num].max(axis=1)
        df['num_min']  = df[all_num].min(axis=1)

    return tr, te

train_fe, test_fe = feature_engineering(train, test)

FEATURES = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]
X        = train_fe[FEATURES]
y        = train_fe[TARGET]
X_test   = test_fe[FEATURES]

print(f'\n✅ Total features : {len(FEATURES)}')
print(f'X      : {X.shape}')
print(f'X_test : {X_test.shape}')

Train : (594194, 21) | Test : (254655, 20)
Churn balance:
Churn
0    0.775
1    0.225
Name: proportion, dtype: float64

Dtypes:
id                    int64
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object
Cat cols (16) : ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod

In [3]:
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb

oof_xgb  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
pred_xgb = np.zeros(len(X_test))
pred_lgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

# ── XGBoost ───────────────────────────────────────────
print('='*50)
print('🚀 Training XGBoost...')
print('='*50)

xgb_params = dict(
    n_estimators          = 3000,
    learning_rate         = 0.02,
    max_depth             = 5,
    min_child_weight      = 10,
    gamma                 = 0.2,
    subsample             = 0.7,
    colsample_bytree      = 0.7,
    reg_alpha             = 0.5,
    reg_lambda            = 2.0,
    scale_pos_weight      = (y==0).sum() / (y==1).sum(),
    tree_method           = 'hist',
    device                = 'cuda',
    eval_metric           = 'auc',
    early_stopping_rounds = 150,
    random_state          = SEED,
    n_jobs                = -1
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              verbose=False)

    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_xgb         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_xgb[val_idx]):.5f}')

print(f'\n📊 XGBoost OOF AUC: {roc_auc_score(y, oof_xgb):.5f}')

# ── LightGBM ──────────────────────────────────────────
print('\n' + '='*50)
print('🚀 Training LightGBM...')
print('='*50)

lgb_params = dict(
    n_estimators      = 3000,
    learning_rate     = 0.02,
    num_leaves        = 31,
    max_depth         = 5,
    min_child_samples = 50,
    feature_fraction  = 0.7,
    bagging_fraction  = 0.7,
    bagging_freq      = 5,
    reg_alpha         = 0.5,
    reg_lambda        = 2.0,
    min_split_gain    = 0.01,
    is_unbalance      = True,
    random_state      = SEED,
    n_jobs            = -1,
    verbosity         = -1
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr,
              eval_set    = [(X_val, y_val)],
              eval_metric = 'auc',
              callbacks   = [
                  lgb.early_stopping(150, verbose=False),
                  lgb.log_evaluation(False)
              ])

    oof_lgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_lgb         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_lgb[val_idx]):.5f}')

print(f'\n📊 LightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.5f}')

# ── CatBoost ──────────────────────────────────────────
print('\n' + '='*50)
print('🚀 Training CatBoost...')
print('='*50)

cat_params = dict(
    iterations         = 3000,
    learning_rate      = 0.02,
    depth              = 5,
    l2_leaf_reg        = 5,
    bootstrap_type     = 'Bernoulli',
    subsample          = 0.7,
    min_data_in_leaf   = 50,
    auto_class_weights = 'Balanced',
    eval_metric        = 'AUC',
    task_type          = 'GPU',
    random_seed        = SEED,
    verbose            = False
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)
    model.fit(X_tr, y_tr,
              eval_set              = (X_val, y_val),
              early_stopping_rounds = 150)

    oof_cat[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_cat         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_cat[val_idx]):.5f}')

print(f'\n📊 CatBoost OOF AUC: {roc_auc_score(y, oof_cat):.5f}')

# ── Summary ───────────────────────────────────────────
print('\n' + '='*50)
print('📊 GBDT Summary')
print('='*50)
print(f'  XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  CatBoost OOF AUC : {roc_auc_score(y, oof_cat):.5f}')

🚀 Training XGBoost...
  Fold  1 | AUC: 0.90225
  Fold  2 | AUC: 0.90139
  Fold  3 | AUC: 0.90389
  Fold  4 | AUC: 0.90047
  Fold  5 | AUC: 0.90182
  Fold  6 | AUC: 0.90268
  Fold  7 | AUC: 0.90304
  Fold  8 | AUC: 0.90192
  Fold  9 | AUC: 0.90099
  Fold 10 | AUC: 0.90093

📊 XGBoost OOF AUC: 0.90192

🚀 Training LightGBM...
  Fold  1 | AUC: 0.89536
  Fold  2 | AUC: 0.89521
  Fold  3 | AUC: 0.89755
  Fold  4 | AUC: 0.89432
  Fold  5 | AUC: 0.89530
  Fold  6 | AUC: 0.89652
  Fold  7 | AUC: 0.89695
  Fold  8 | AUC: 0.89598
  Fold  9 | AUC: 0.89470
  Fold 10 | AUC: 0.89457

📊 LightGBM OOF AUC: 0.89550

🚀 Training CatBoost...


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  1 | AUC: 0.90074


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  2 | AUC: 0.90054


...

  Fold  8 | AUC: 0.90105


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  9 | AUC: 0.89977


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 10 | AUC: 0.89953

📊 CatBoost OOF AUC: 0.90073

📊 GBDT Summary
  XGBoost  OOF AUC : 0.90192
  LightGBM OOF AUC : 0.89550
  CatBoost OOF AUC : 0.90073


In [4]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.linear_model import LogisticRegression

# ── Stack OOFs ────────────────────────────────────────
names       = ['XGBoost', 'LightGBM', 'CatBoost']
oof_matrix  = np.column_stack([oof_xgb, oof_lgb, oof_cat])
pred_matrix = np.column_stack([pred_xgb, pred_lgb, pred_cat])

# ── Optuna Weight Search ──────────────────────────────
print('🔍 Running Optuna weight search (1000 trials)...')

def objective(trial):
    w = np.array([trial.suggest_float(f'w{i}', 0.0, 1.0) for i in range(3)])
    w = w / w.sum()
    return roc_auc_score(y, (oof_matrix * w).sum(axis=1))

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000, show_progress_bar=True)

best_w = np.array([study.best_params[f'w{i}'] for i in range(3)])
best_w = best_w / best_w.sum()

print(f'\n⚖️  Optimal weights:')
for n, w in zip(names, best_w):
    print(f'  {n:12s}: {w:.4f}')

oof_blend  = (oof_matrix  * best_w).sum(axis=1)
pred_blend = (pred_matrix * best_w).sum(axis=1)
print(f'\n📊 Weighted Blend OOF AUC : {roc_auc_score(y, oof_blend):.5f}')

# ── Level-2 Stacker ───────────────────────────────────
print('\n🚀 Training Level-2 Stacker...')

oof_stack  = np.zeros(len(y))
pred_stack = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_matrix, y)):
    meta = LogisticRegression(C=1.0, max_iter=1000)
    meta.fit(oof_matrix[tr_idx], y.iloc[tr_idx])
    oof_stack[val_idx]  = meta.predict_proba(oof_matrix[val_idx])[:, 1]
    pred_stack         += meta.predict_proba(pred_matrix)[:, 1] / N_SPLITS

print(f'📊 Level-2 Stack OOF AUC  : {roc_auc_score(y, oof_stack):.5f}')

# ── Final Blend ───────────────────────────────────────
FINAL_W    = 0.6
oof_final  = FINAL_W * oof_blend  + (1 - FINAL_W) * oof_stack
pred_final = FINAL_W * pred_blend + (1 - FINAL_W) * pred_stack

print(f'\n🏆 FINAL OOF AUC : {roc_auc_score(y, oof_final):.5f}')

# ── Submit ────────────────────────────────────────────
sub['Churn'] = pred_final
sub.to_csv('submission.csv', index=False)

print(f'\n✅ submission.csv saved — {len(sub)} rows')
print(f'  Mean : {pred_final.mean():.4f}')
print(f'  Std  : {pred_final.std():.4f}')
print(f'  Min  : {pred_final.min():.4f}')
print(f'  Max  : {pred_final.max():.4f}')

# ── Final Summary ─────────────────────────────────────
print('\n' + '='*50)
print('🏆 FINAL SUMMARY')
print('='*50)
print(f'  XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  CatBoost OOF AUC : {roc_auc_score(y, oof_cat):.5f}')
print(f'  Blend    OOF AUC : {roc_auc_score(y, oof_blend):.5f}')
print(f'  Stack    OOF AUC : {roc_auc_score(y, oof_stack):.5f}')
print(f'  ─────────────────────────────')
print(f'  FINAL    OOF AUC : {roc_auc_score(y, oof_final):.5f} 🎯')
print('='*50)
print('\n✅ DONE — Output tab → submission.csv → Submit!')

🔍 Running Optuna weight search (1000 trials)...


  0%|          | 0/1000 [00:00<?, ?it/s]


⚖️  Optimal weights:
  XGBoost     : 0.9653
  LightGBM    : 0.0000
  CatBoost    : 0.0347

📊 Weighted Blend OOF AUC : 0.90192

🚀 Training Level-2 Stacker...
📊 Level-2 Stack OOF AUC  : 0.90193

🏆 FINAL OOF AUC : 0.90193

✅ submission.csv saved — 254655 rows
  Mean : 0.2998
  Std  : 0.3031
  Min  : 0.0060
  Max  : 0.9237

🏆 FINAL SUMMARY
  XGBoost  OOF AUC : 0.90192
  LightGBM OOF AUC : 0.89550
  CatBoost OOF AUC : 0.90073
  Blend    OOF AUC : 0.90192
  Stack    OOF AUC : 0.90193
  ─────────────────────────────
  FINAL    OOF AUC : 0.90193 🎯

✅ DONE — Output tab → submission.csv → Submit!
